# TASK 16 ? Samsung Health: kelayakan dan rancangan model

## Ringkasan
Pelatihan **ditunda**: laporan integrasi masih menandai semantik waktu, batas hari aktivitas dan satuan sebagai kandidat. Pemilihan sesi terpanjang secara retrospektif belum memenuhi target sesi berikutnya. Notebook menjalankan EDA privat dan memverifikasi pengunci pelatihan; tidak menghasilkan metrik model rekaan.

Salinan publik harus bebas output. Hasil eksekusi dan laporan kesehatan disimpan lokal dalam direktori yang diabaikan Git.

## Business Understanding dan metode
Tujuan: memprediksi `sleep_duration_minutes` pada sesi berikutnya. Dummy median, Linear Regression, dan Random Forest direncanakan; tidak ada dugaan model terbaik.

### Asumsi utama
`date` adalah tanggal bangun Jakarta. Diagnostik aktivitas memakai tanggal UTC mulai tidur dikurangi satu hari, dengan ringkasan hari selesai sebelum awal tidur. Ini hanya aturan konservatif retrospektif, bukan bukti tersedia pada saat prediksi. Perlu seluruh sesi dan aturan target yang ditetapkan sebelum pengamatan.

Setelah data layak: split kronologis 80/20, purge label melintasi split, validasi temporal pada train bila cakupan memadai. Imputasi/scaling hanya fit di Pipeline tiap fold. Tuning RF dibatasi pada train; semua model menggunakan test sama. MAE utama, RMSE dan R? pelengkap. Belum diimplementasikan sebagai evaluasi final karena gate data gagal.

## Setup lokal dan Google Colab
Lokal: install `requirements-modeling.txt` sekali per environment lalu pilih kernel environment tersebut.
Colab setiap runtime: simpan kode repo dan CSV privat di Drive, mount Drive, sesuaikan `PROJECT_ROOT`, lalu install requirements. Tidak ada data kesehatan yang diunduh dari URL publik. Jalankan dari atas hingga bawah.

Hapus output sebelum commit. Jangan membagikan folder `artifacts/` atau laporan privat.

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/uas-tasya-ds")  # Sesuaikan sekali.
else:
    PROJECT_ROOT = Path.cwd()
    if not (PROJECT_ROOT / "src").is_dir():
        PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "src/modeling").is_dir(), "Sediakan kode repo di PROJECT_ROOT"
if IN_COLAB:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "-r", str(PROJECT_ROOT / "requirements-modeling.txt")])
sys.path.insert(0, str(PROJECT_ROOT))
SOURCE = PROJECT_ROOT / "data_processed/samsung_health_merged.csv"
OUTPUT = PROJECT_ROOT / "artifacts/task16"
REPORT = PROJECT_ROOT / "reports/model_evaluation.md"
assert SOURCE.is_file(), "Dataset privat belum tersedia; jangan gunakan demo sebagai pengganti"


## Data Understanding ? sumber dan audit
Baca laporan integrasi sebelum menafsirkan CSV. `cleaning_report.md` bersifat opsional; cleaning juga didokumentasikan dalam laporan integrasi. Audit memeriksa skema, nilai, missing, duplikasi, kalender dan offset eksplisit. Statistik berikut bersifat privat.

In [ ]:
from IPython.display import Image, Markdown, display

from src.modeling.eda import write_diagnostics
from src.modeling.readiness import load_candidate, require_ready

integration = PROJECT_ROOT / "reports/dataset_integration_report.md"
assert integration.is_file(), "Laporan integrasi diperlukan"
assert integration.read_text(encoding="utf-8").strip(), "Laporan integrasi kosong"
cleaning = PROJECT_ROOT / "reports/cleaning_report.md"
print("Laporan cleaning terpisah tersedia:", cleaning.is_file())
frame = load_candidate(SOURCE)
audit = write_diagnostics(SOURCE, OUTPUT, REPORT)
display({k: audit[k] for k in ["status", "rows", "columns", "five_source_fields_complete",
                              "retrospective_complete_pairs", "effective_training_samples"]})
display(frame.dtypes.astype(str).to_frame("dtype"))
display(frame.isna().sum().to_frame("missing"))
print("Penghambat:", *audit["blockers"], sep="\n- ")


## EDA ? distribusi, aktivitas, pola hari, korelasi dan outlier
EDA ini memakai label kandidat untuk deskripsi. Korelasi pairwise pada tanggal sama bukan hubungan aktivitas dengan tidur berikutnya dan bukan sebab-akibat. Flag 1,5?IQR tidak dihapus otomatis; nol langkah tidak membuktikan perangkat dipakai. Missing target tidak diimputasi.

In [ ]:
import pandas as pd

display(frame[["sleep_duration_minutes", "step_count", "active_time_minutes"]].describe())
display(audit["iqr_outlier_counts"])
display(pd.read_csv(OUTPUT / "day_patterns.csv", header=[0, 1], index_col=0))
for name in ["sleep_distribution", "activity_distribution", "day_patterns",
             "correlations", "missing_values"]:
    display(Image(filename=str(OUTPUT / f"{name}.png")))


## Modeling ? spesifikasi dan gate
Fitur final belum disahkan. Kandidat: langkah, menit aktif, kalender waktu prediksi, durasi dan jam tidur sebelumnya yang sudah selesai sebelum prediksi. Jam historis perlu sin/cos; `distance_km` tidak tersedia. Jangan memasukkan akhir tidur, jam bangun, interval atau bedtime target.

Pipeline berikut **hanya dibuat, tidak di-fit**. Gate sengaja tertutup sampai kontrak data diperbaiki berdasarkan bukti. Tidak ada flag untuk mengabaikan penghambat.

In [ ]:
from src.modeling.models import model_specs

specs = model_specs()
print("Model direncanakan:", list(specs))
try:
    require_ready(audit)
except ValueError as exc:
    print(str(exc))
else:
    raise RuntimeError("Gate kandidat tidak boleh terbuka tanpa revisi kontrak")
assert audit["train_samples"] == audit["test_samples"] == 0


## Evaluation dan temuan
Laporan berikut dibuat dari audit dan EDA yang sama, sehingga jumlah sampel dan status konsisten. MAE, RMSE, R?, actual vs predicted, residual, importance dan analisis error belum tersedia. Tidak ada model terbaik. K-Means ditunda karena semantik waktu dan satuan belum terverifikasi; tidak ada silhouette yang dihitung.

In [ ]:
display(Markdown(REPORT.read_text(encoding="utf-8")))
assert "Sampel efektif untuk pelatihan: **0" in REPORT.read_text(encoding="utf-8")
assert all(not hasattr(p.named_steps["imputer"], "statistics_") for p in specs.values())
print("Diagnostik selesai; tidak ada model dilatih.")


## Takeaways dan Deployment
Hasil diagnostik tidak memenuhi prasyarat pelatihan prospektif. Tindak lanjut: konfirmasi waktu/satuan dari aplikasi atau sumber yang sah; tetapkan sesi target sebelum pengamatan; bentuk ulang fitur dengan waktu ketersediaan eksplisit. Selanjutnya implementasikan evaluasi temporal dan bandingkan dengan baseline secara objektif.

Data satu individu, missing, perangkat dan seleksi sesi membatasi generalisasi dan tidak mendukung klaim klinis. Dashboard publik hanya menampilkan status metode. Eksekusi lokal tidak memverifikasi mount Drive atau render Colab. Salinan publik tetap tanpa output kesehatan pribadi.